# 01 - Attention from scratch

**What:** build causal self-attention as the bare mathematics - QK^T, scale, mask, softmax, AV - nothing fused.

**Why:** the plan (section 30) requires the naked implementation BEFORE the optimized one. If you can't write attention yourself, you can't claim to know what FlashAttention is replacing.

**How:** every step below prints its shape; the last cell proves our manual result equals the lab's `attention_vanilla`.

In [ ]:
# --- locate the repo (works locally AND on Colab/Kaggle pasted into a
# fresh notebook: it finds an existing copy, pulls the latest version from
# GitHub, or clones if missing) -------------------------------------------
import os, sys, subprocess

REPO_URL = "https://github.com/Th3Samaritan/nano-gpt-lab.git"

def _find_repo():
    here = os.path.abspath("")
    candidates = [here, os.path.dirname(here),
                  os.path.join(here, "nano-gpt-lab"),
                  "/kaggle/working/nano-gpt-lab",
                  "/content/nano-gpt-lab"]
    for d in candidates:
        if d and os.path.exists(os.path.join(d, "scripts", "train.py")):
            return d
    return None

repo = _find_repo()
if repo is None:
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        dest = "/kaggle/working/nano-gpt-lab"
    elif "google.colab" in sys.modules:
        dest = "/content/nano-gpt-lab"
    else:
        dest = os.path.join(os.path.abspath(""), "nano-gpt-lab")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, dest], check=True)
    repo = dest
elif os.path.isdir(os.path.join(repo, ".git")):
    # Already have a copy: bring it up to date. --ff-only means the update
    # can only fast-forward, so an edited/runtime copy is never mangled by
    # a merge; on failure we keep the existing code and continue.
    try:
        r = subprocess.run(["git", "-C", repo, "pull", "--ff-only"],
                           capture_output=True, text=True, timeout=120)
        print((r.stdout or r.stderr or "already up to date").strip())
    except Exception as e:
        print("git pull skipped:", e)
    # Drop any src modules cached from before the pull so the fresh code
    # is used, not the stale in-memory copy.
    for mod in list(sys.modules):
        if mod == "src" or mod.startswith("src."):
            del sys.modules[mod]
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")  # Windows OMP clash
sys.path.insert(0, repo)
os.chdir(repo)
print("repo on path:", repo)

In [ ]:
import torch, math
import torch.nn.functional as F

# --- fake inputs: batch 2, heads 4, 32 tokens, 16-dim per head ------------
torch.manual_seed(1)
B, H, T, D = 2, 4, 32, 16
X = torch.randn(B, T, H * D)                    # hidden states (upstream of us)

# --- learnable projections: Q = X Wq, K = X Wk, V = X Wv ------------------
Wq = torch.randn(H * D, H * D) * 0.02
Wk = torch.randn(H * D, H * D) * 0.02
Wv = torch.randn(H * D, H * D) * 0.02
Q = X @ Wq
K = X @ Wk
V = X @ Wv

# --- split heads: (B, H, T, D) --------------------------------------------
Q = Q.view(B, T, H, D).transpose(1, 2)
K = K.view(B, T, H, D).transpose(1, 2)
V = V.view(B, T, H, D).transpose(1, 2)
print("Q shape:", tuple(Q.shape), " K:", tuple(K.shape), " V:", tuple(V.shape))

In [ ]:
# --- 1. SCORES: how much every pair of tokens 'agrees' -------------------
scores = Q @ K.transpose(-2, -1)          # (B, H, T, T): the T x T table
print("scores shape:", tuple(scores.shape), "(the big T x T table)")

# --- 2. SCALE: divide by sqrt(d) so softmax doesn't saturate ---------------
scale = D ** -0.5
scores = scores * scale

# --- 3. CAUSAL MASK: a token may only look BACK ---------------------------
#     Lower-triangle mask: row i keeps columns 0..i, blocks i+1..T-1
mask = torch.tril(torch.ones(T, T, dtype=torch.bool))[None, None]
scores = scores.masked_fill(~mask, float("-inf"))
print("mask: True = allowed, False = blocked")
print(mask[0, 0, 0].int())  # first row: only the very first token allowed

In [ ]:
# --- 4. SOFTMAX: scores -> percentages (each row sums to 1) ---------------
att = F.softmax(scores, dim=-1)           # (B, H, T, T)
print("row 10 sums to:", att[0, 0, 10].sum().item(), "(must be 1.0)")
print("first 6 weights of row 10:", att[0, 0, 10, :6].tolist())

# --- 5. GATHER: blend the values V by those percentages --------------------
Y = att @ V                               # (B, H, T, D)
print("Y shape:", tuple(Y.shape))

# --- visual: attention weights of one head --------------------------------
import matplotlib.pyplot as plt
plt.figure(figsize=(6, 5))
plt.imshow(att[0, 0].detach().numpy(), cmap="viridis", aspect="auto")
plt.colorbar(label="attention weight")
plt.xlabel("token looked AT (past)"); plt.ylabel("token doing the looking")
plt.title("Attention map: lower-triangular = causal (no peeking ahead)")
plt.show()

In [ ]:
# --- VALIDATION (plan section 31): our math vs the lab implementation -----
from src.model.attention import attention_vanilla

y_lab = attention_vanilla(Q, K, V, torch.tril(torch.ones(T, T, dtype=torch.bool))[None, None])
diff = (Y - y_lab).abs().max().item()
print("max |mine - lab| =", diff)
assert diff < 1e-6, "mismatch!"
print("EXACT MATCH: the naked math above IS the lab's vanilla attention.")

**Takeaway:** attention = compare, normalize, gather. The `T x T` scores table is the only thing in the whole model that costs O(T^2) memory - that table is exactly what FlashAttention avoids (see notebook 04's charts and src/model/flash_attention.py).